## Example 1 — suffix decoding: a request's life (cache a response, then draft from it)

In [ ]:
import numpy as np
from arctic_inference.suffix_decoding import SuffixDecodingCache, _C
assert _C.__file__.endswith('.so'), _C.__file__
cache = SuffixDecodingCache(max_tree_depth=64, max_cached_requests=8)
prompt = np.arange(1, 9, dtype=np.int32)
cache.start_request('first', prompt)
cache.add_active_response('first', [10, 11, 12, 13, 14, 15, 16, 17])
assert list(cache.active_requests) == ['first']
cache.stop_request('first')  # the response stays in the global cache
assert list(cache.active_requests) == [] and list(cache.cached_requests) == ['first']
cache.start_request('second', prompt)
# A context that ends like the cached response: the draft is that response's continuation.
# (max_spec_tokens is always given, as vLLM does.)
draft = cache.speculate('second', np.array([1, 2, 3, 10, 11, 12], dtype=np.int32), max_spec_tokens=3)
assert draft.token_ids == [13, 14, 15] and draft.match_len == 3, draft
assert draft.parents == [-1, 0, 1] and draft.score == 3.0 and draft.probs == [1.0, 1.0, 1.0], draft
context = np.array([10, 11, 12], dtype=np.int32)
short = cache.speculate('second', context, max_spec_tokens=4)
longer = cache.speculate('second', context, max_spec_tokens=4, max_spec_factor=2.0)
assert short.token_ids == [13, 14] and longer.token_ids == [13, 14, 15, 16], (short, longer)
assert longer.match_len >= 2 and longer.score == 4.0, longer
nothing = cache.speculate('second', [77, 78], max_spec_tokens=8)  # a context never seen
assert nothing.token_ids == [] and nothing.score == 0.0, nothing
cache.stop_request('second')
print('OK -- after [.., 10, 11, 12] the cache proposes', draft.token_ids, '; a factor of 2.0 turns', short.token_ids, 'into',
      longer.token_ids)

## Example 2 — the vLLM plug-in is registered and stays switched off; wrong input is refused

In [ ]:
import importlib.metadata, os
import numpy as np
import arctic_inference.envs as envs
from arctic_inference.suffix_decoding import SuffixDecodingCache
(entry_point,) = [ep for ep in importlib.metadata.entry_points(group='vllm.general_plugins') if ep.name == 'arctic_inference']
assert entry_point.value == 'arctic_inference.vllm.plugin:arctic_inference_plugin', entry_point.value
assert os.environ.get('ARCTIC_INFERENCE_ENABLED', '0') != '1'
assert entry_point.load()() is None
assert envs.arctic_inference_effective_enabled() is False
assert envs.arctic_inference_effective_enabled({'ARCTIC_INFERENCE_ENABLED': '1'}) is True  # a worker's own setting
# Wrong input to the cache is refused with a ValueError before it reaches the compiled suffix tree.
cache = SuffixDecodingCache(max_tree_depth=8)
cache.start_request('a', [1, 2, 3])
refusals = []
for call in (lambda: cache.start_request('a', [1]),                                  # already active
             lambda: cache.speculate('never started', [1], max_spec_tokens=1),       # not active
             lambda: cache.speculate('a', np.array([1.0, 2.0]), max_spec_tokens=1)):  # not int32
    try:
        call()
        raise AssertionError('a wrong call was accepted')
    except ValueError as error:
        refusals.append(str(error))
assert 'already active' in refusals[0] and 'not active' in refusals[1] and 'int32' in refusals[2], refusals
print('OK -- the vLLM plug-in is registered and inert; 3 wrong calls raise ValueError')

## Example 3 — drafting from the request's own prompt, and a tree of alternatives

In [ ]:
from arctic_inference.suffix_decoding import SuffixDecodingCache

cache = SuffixDecodingCache(max_tree_depth=16)
cache.start_request("copy", [1, 2, 3, 4, 5, 6, 7, 8])
cache.add_active_response("copy", [4, 5])  # the model starts repeating its prompt
draft = cache.speculate("copy", [3, 4, 5], max_spec_tokens=3, max_spec_factor=4.0)
assert draft.token_ids == [6, 7, 8], draft  # continued from the prompt, nothing cached yet
cache.stop_request("copy")
votes = SuffixDecodingCache(max_tree_depth=16)
for request, response in enumerate(([1, 2, 3, 4], [1, 2, 3, 5], [1, 2, 3, 4])):
    votes.start_request(request, [])
    votes.add_active_response(request, response)
    votes.stop_request(request)
votes.start_request("ask", [])
single = votes.speculate("ask", [1, 2, 3], max_spec_tokens=4)
tree = votes.speculate("ask", [1, 2, 3], max_spec_tokens=4, use_tree_spec=True)
assert single.token_ids == [4] and abs(single.probs[0] - 2 / 3) < 1e-6, single
assert tree.token_ids == [4, 5] and tree.parents == [-1, -1], tree
assert abs(tree.probs[0] - 2 / 3) < 1e-6 and abs(tree.probs[1] - 1 / 3) < 1e-6, tree
print("OK -- prompt copy gives", draft.token_ids, "; after [1, 2, 3] the tree offers 4 (2/3) and 5 (1/3)")

## Example 4 — the cache keeps the newest requests only (first in, first out)

In [ ]:
from arctic_inference.suffix_decoding import SuffixDecodingCache

cache = SuffixDecodingCache(max_tree_depth=32, max_cached_requests=4)
assert (cache.max_tree_depth, cache.max_cached_requests) == (32, 4)
for request in range(6):
    cache.start_request(request, [100 + request])
    cache.add_active_response(request, [200 + request, 300 + request, 400 + request])
    cache.stop_request(request)
assert list(cache.cached_requests) == [2, 3, 4, 5], list(cache.cached_requests)
cache.start_request("probe", [])  # a new request takes a slot at once, so request 2 goes too
assert list(cache.cached_requests) == [3, 4, 5, "probe"], list(cache.cached_requests)
proposals = {r: cache.speculate("probe", [200 + r, 300 + r], max_spec_tokens=2).token_ids for r in range(6)}
assert proposals == {0: [], 1: [], 2: [], 3: [403], 4: [404], 5: [405]}, proposals
cache.stop_request("probe")
cache.evict_cached_response(5)
assert list(cache.cached_requests) == [3, 4, "probe"], list(cache.cached_requests)
unlimited = SuffixDecodingCache(max_tree_depth=32)  # -1: no limit
for request in range(6):
    unlimited.start_request(request, [])
    unlimited.stop_request(request)
assert list(unlimited.cached_requests) == list(range(6))
print("OK -- a cache of 4 keeps the newest responses only:", proposals)